# Туториал: In-Memory режим (`streaming.mode=none`)

Цель: быстро запустить пайплайн, где финальные `SharedSample` идут через in-memory cache.

Подходит для:
- локальной отладки
- smoke-проверок
- коротких экспериментов


In [ ]:
from __future__ import annotations

import sys
import platform
from pathlib import Path
from collections import Counter

import torch
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from dataset.shared.collector_service import CollectorService
from dataset.shared.shared_dataset import SharedModelDataset

def load_hydra_cfg(config_path: str = "conf/config.yaml", overrides: list[str] | None = None):
    cfg_path = repo_root / config_path
    with initialize_config_dir(version_base=None, config_dir=str(cfg_path.parent.resolve())):
        return compose(config_name=cfg_path.stem, overrides=overrides or [])

print(f'Python: {platform.python_version()}')
print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device count: {torch.cuda.device_count()}')


## Минимальная конфигурация

Выберите датасеты и модели через override'ы.


In [ ]:
DATA_ROOT = "./data"

overrides = [
    f"data.path={DATA_ROOT}",
    "streaming.mode=none",
    "collector.mode=auto",
    "collector.device=null",
    "data.enabled_datasets=[coco2017,cc12m]",
    "data.dataset_overrides.coco2017.models=[clip_vit_b32]",
    "data.dataset_overrides.cc12m.models=[clip_vit_b32]",
    "collector.cache.max_items=128",
    "collector.cache.fill_target=128",
    "collector.cache.low_watermark=64",
    "collector.interleaved.every_n_steps=5",
    "collector.interleaved.burst_jobs=1",
    "collector.atomization.chunk_rows=64",
]

cfg = load_hydra_cfg(overrides=overrides)
print(OmegaConf.to_yaml(cfg.collector, resolve=True))
print('streaming mode =', cfg.streaming.mode)


In [ ]:
collector = CollectorService(cfg)
dataset = SharedModelDataset(collector)

collector.start()

model_counts = Counter()
layer_counts = Counter()
consumed = 0

try:
    for step in range(80):
        dataset.maybe_collect(step)
        sample = dataset.try_next_sample()
        if sample is None:
            continue
        consumed += 1
        model_counts[sample.model_name] += 1
        layer_counts[sample.layer_name] += 1

    print('consumed:', consumed)
    print('cache_size:', dataset.cache_size())
    print('model_counts:', dict(model_counts))
    print('top_layers:', layer_counts.most_common(5))
finally:
    dataset.close()
    collector.shutdown()


## Troubleshooting

- Ошибка token: проверьте `hf.token` в `conf/config.yaml`.
- Нет сэмплов: проверьте `data.enabled_datasets` и `models` в dataset YAML.
- Медленно: уменьшите `model.batch_size`, `collector.atomization.chunk_rows`.
